# Compare SDXL Turbo and SDXL Base

Enter one prompt and generate a small batch with both models. Each model gets the same list of numeric seeds and uses its standard sampling profile. The images are for a visual comparison: Turbo and Base have different resolutions and sampling behavior, so the same seed does not mean identical starting noise across both models.

Run from the debiasing repository on a CUDA GPU. The first run may download model weights from Hugging Face.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "scripts/generate/generation_utils.py").exists())
sys.path.insert(0, str(ROOT))

import torch
import ipywidgets as widgets
from IPython.display import display, clear_output
from scripts.generate.generation_utils import load_pipeline, generate_batch

if not torch.cuda.is_available():
    raise RuntimeError("This notebook currently needs a CUDA GPU. Start Jupyter on a GPU node.")


## Configure and run

Start with a small count (1–4) because this loads each large model in turn. The notebook releases one model before loading the other.

In [ ]:
prompt = widgets.Textarea(value="A photo of a flower in a garden", description="Prompt:", layout=widgets.Layout(width="90%", height="90px"))
count = widgets.IntSlider(value=2, min=1, max=6, step=1, description="Images/model:")
seed = widgets.IntText(value=0, description="First seed:")
button = widgets.Button(description="Generate both", button_style="primary", icon="images")
output = widgets.Output()

def on_compare(_):
    with output:
        clear_output(wait=True)
        if not prompt.value.strip():
            print("Enter a prompt first.")
            return
        for model_name in ["SDXL Turbo", "SDXL Base"]:
            pipe = None
            try:
                print(f"Loading {model_name}…")
                pipe = load_pipeline(model_name)
                defaults = {"SDXL Turbo": (1, 0.0), "SDXL Base": (30, 5.0)}[model_name]
                images, run_dir, _ = generate_batch(
                    pipe=pipe,
                    model_name=model_name,
                    prompt=prompt.value,
                    count=count.value,
                    seed=seed.value,
                    steps=defaults[0],
                    guidance=defaults[1],
                    output_root=ROOT / "outputs/interactive_generation",
                    run_name="model_comparison_" + model_name.lower().replace(" ", "_"),
                )
                print(f"{model_name} — saved to {run_dir}")
                display(*images)
            except Exception as exc:
                print(f"{model_name} failed: {type(exc).__name__}: {exc}")
            finally:
                if pipe is not None:
                    del pipe
                torch.cuda.empty_cache()

button.on_click(on_compare)
display(prompt, count, seed, button, output)


The batch folders contain `settings.json`, including each model ID, sampling profile, prompt, and seed list. For a controlled SAE experiment, keep those settings with the run and separately train SAE features on the activation distribution of the chosen model.